In [1]:

import numpy as np
import pandas as pd

In [2]:
books=pd.read_csv('Books.csv')
users=pd.read_csv('Users.csv')
ratings=pd.read_csv('Ratings.csv')


C:\Users\hp\AppData\Local\Temp\ipykernel_18008\3430967063.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books=pd.read_csv('Books.csv')


In [3]:
print(books.shape)
print(users.shape)
print(ratings.shape)

(271360, 8)
(278858, 3)
(1149780, 3)


In [4]:
books.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [5]:
users.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [6]:
books.duplicated().sum()

np.int64(0)

In [7]:
ratings.head()


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [8]:
#performing normalization on the data, to remove bias from user ratings
ratings['Book-Rating'].value_counts(normalize=True)
#i am checking for how many ratings are given per user, those users who have rated less books i will remove, becuase they crate uncessary noise
ratings_count=ratings.groupby('User-ID')['Book-Rating'].count()
print(ratings_count)
valid_users=ratings_count[ratings_count>=5].index


User-ID
2          1
7          1
8         18
9          3
10         2
          ..
278846     2
278849     4
278851    23
278852     1
278854     8
Name: Book-Rating, Length: 105283, dtype: int64


In [9]:
filtered_ratings = ratings[ratings['User-ID'].isin(valid_users)]


In [10]:
num_users=ratings['User-ID'].nunique()
num_books=ratings['ISBN'].nunique()
num_ratings=len(ratings)
sparsity=1-(num_ratings/(num_users*num_books))

print('Sparsity',sparsity)

Sparsity 0.9999679322889055


In [11]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [12]:
books=books.dropna(subset=['ISBN'])
books=books.drop_duplicates(subset='ISBN')
#stripping whitespaces
books['title']=books['Book-Title'].str.strip()
books['author']=books['Book-Author'].str.strip()

In [13]:
#filter books with very few ratings

books_counts=filtered_ratings.groupby('ISBN')['Book-Rating'].count()
valid_books=books_counts[books_counts>=5].index
#now filtered ratings contains only those books and users who are active
filtered_ratings=filtered_ratings[filtered_ratings['ISBN'].isin(valid_books)]

In [14]:
#recalculate sparsity i am checking sparsity after every step of cleaning to check if its still high, here the sparsity is still really high so ill consider filtering or matrix factorization
num_users = filtered_ratings['User-ID'].nunique()
num_books = filtered_ratings['ISBN'].nunique()
num_ratings = len(filtered_ratings)

sparsity = 1 - (num_ratings / (num_users * num_books))
print(f"Sparsity: {sparsity:.4f}")


Sparsity: 0.9993


In [15]:
#the filtered books have all the books i have kept after i removed the users who gave less ratings and books with less amount of ratings
filtered_books = books[books['ISBN'].isin(filtered_ratings['ISBN'])]



In [16]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [17]:
#cleaning the users dataset 
users=users.drop_duplicates(subset='User-ID')
users['Age']=users["Age"].fillna(users['Age'].median())
users['Location']=users['Location'].fillna('Unknown')

In [18]:
#merge ratings with book and user metadata
merged=filtered_ratings.merge(filtered_books,on='ISBN',how='left')
merged=merged.merge(users,on='User-ID',how='left')

In [19]:
#build ratin matrix
rating_matrix = merged.pivot(index='User-ID', columns='ISBN', values='Book-Rating')



In [20]:
print(rating_matrix.shape)
print(filtered_books.shape)
print(users.shape)



(21915, 39702)
(37536, 10)
(278858, 3)


In [21]:
#other way, from youtube
ratings_with_name=ratings.merge(books,on='ISBN')

In [22]:
ratings_with_name

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,title,author
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,Flesh Tones: A Novel,M. J. Rose
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,Rites of Passage,Judith Rae
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,http://images.amazon.com/images/P/0446520802.0...,The Notebook,Nicholas Sparks
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,Help!: Level 1,Philip Prowse
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather
...,...,...,...,...,...,...,...,...,...,...,...,...
1031131,276704,0876044011,0,Edgar Cayce on the Akashic Records: The Book o...,Kevin J. Todeschi,1998,A.R.E. Press (Association of Research &amp; Enlig,http://images.amazon.com/images/P/0876044011.0...,http://images.amazon.com/images/P/0876044011.0...,http://images.amazon.com/images/P/0876044011.0...,Edgar Cayce on the Akashic Records: The Book o...,Kevin J. Todeschi
1031132,276704,1563526298,9,Get Clark Smart : The Ultimate Guide for the S...,Clark Howard,2000,Longstreet Press,http://images.amazon.com/images/P/1563526298.0...,http://images.amazon.com/images/P/1563526298.0...,http://images.amazon.com/images/P/1563526298.0...,Get Clark Smart : The Ultimate Guide for the S...,Clark Howard
1031133,276706,0679447156,0,Eight Weeks to Optimum Health: A Proven Progra...,Andrew Weil,1997,Alfred A. Knopf,http://images.amazon.com/images/P/0679447156.0...,http://images.amazon.com/images/P/0679447156.0...,http://images.amazon.com/images/P/0679447156.0...,Eight Weeks to Optimum Health: A Proven Progra...,Andrew Weil
1031134,276709,0515107662,10,The Sherbrooke Bride (Bride Trilogy (Paperback)),Catherine Coulter,1996,Jove Books,http://images.amazon.com/images/P/0515107662.0...,http://images.amazon.com/images/P/0515107662.0...,http://images.amazon.com/images/P/0515107662.0...,The Sherbrooke Bride (Bride Trilogy (Paperback)),Catherine Coulter


In [23]:
#popularity based recommender system
#we find how many votes each group got
num_rating_dataframe=ratings_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_dataframe.rename(columns={'Book-Rating':'num_ratings'},inplace=True)
num_rating_dataframe

,Book-Title,num_ratings
0,A Light in the Storm: The Civil War Diary of ...,4
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1
4,Beyond IBM: Leadership Marketing and Finance ...,1
...,...,...
241066,Ã?Â?lpiraten.,2
241067,Ã?Â?rger mit Produkt X. Roman.,4
241068,Ã?Â?sterlich leben.,1
241069,Ã?Â?stlich der Berge.,3


In [24]:
#avg ratings of each book
avg_rating_dataframe=ratings_with_name.groupby('Book-Title')['Book-Rating'].mean().reset_index()
avg_rating_dataframe.rename(columns={'Book-Rating':'avg_ratings'},inplace=True)
avg_rating_dataframe

,Book-Title,avg_ratings
0,A Light in the Storm: The Civil War Diary of ...,2.250000
1,Always Have Popsicles,0.000000
2,Apple Magic (The Collector's series),0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,0.000000
...,...,...
241066,Ã?Â?lpiraten.,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,5.250000
241068,Ã?Â?sterlich leben.,7.000000
241069,Ã?Â?stlich der Berge.,2.666667


In [25]:
popular_df=num_rating_dataframe.merge(avg_rating_dataframe,on='Book-Title')
popular_df

,Book-Title,num_ratings,avg_ratings
0,A Light in the Storm: The Civil War Diary of ...,4,2.250000
1,Always Have Popsicles,1,0.000000
2,Apple Magic (The Collector's series),1,0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1,8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,1,0.000000
...,...,...,...
241066,Ã?Â?lpiraten.,2,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,4,5.250000
241068,Ã?Â?sterlich leben.,1,7.000000
241069,Ã?Â?stlich der Berge.,3,2.666667


In [26]:
popular_df=popular_df[popular_df['num_ratings']>=250].sort_values('avg_ratings',ascending=False)

In [27]:
#we have created the popular dataframe now
#we want the popular books dataframe to have other details too so we are going ot merge
popular_df=popular_df.merge(books,on='Book-Title').drop_duplicates('Book-Title')[['Book-Title','author','Image-URL-M','num_ratings','avg_ratings']]
popular_df


,Book-Title,author,Image-URL-M,num_ratings,avg_ratings
0,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,http://images.amazon.com/images/P/0439136350.0...,428,5.852804
3,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,http://images.amazon.com/images/P/0439139597.0...,387,5.824289
5,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,http://images.amazon.com/images/P/0590353403.0...,278,5.737410
9,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,http://images.amazon.com/images/P/043935806X.0...,347,5.501441
13,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,http://images.amazon.com/images/P/0439064872.0...,556,5.183453
...,...,...,...,...,...
716,Vinegar Hill (Oprah's Book Club (Paperback)),A. Manette Ansay,http://images.amazon.com/images/P/0380730138.0...,265,2.245283
717,Whispers,BELVA PLAIN,http://images.amazon.com/images/P/0440216745.0...,286,2.199301
727,Presumed Innocent,Scott Turow,http://images.amazon.com/images/P/0446359866.0...,294,2.139456
733,Isle of Dogs,Patricia Cornwell,http://images.amazon.com/images/P/0425182908.0...,288,2.000000


In [28]:
#now we are going tp the make the recommender system which is collaborative filtering based
#it will rely on how much ratings the books got, we will use the users who have given more ratings on book and use books who had more ratings and were more read so basically active users and books approach


x=ratings_with_name.groupby('User-ID').count()['Book-Rating']>200
padhe_likhe_users=x[x].index
#it will display the user ids of people who have given more than 200 book ratings
filtered_ratings=ratings_with_name[ratings_with_name['User-ID'].isin(padhe_likhe_users)]
#now what we have done is that we have filtered tha dataframe and only kept the users which had more ratings
#maine usko bola k us table mein sey sirf unki user id rakho jo padhe likhe users mein thay, baki sab k baray mein idgaf



In [29]:
#ab we are going to filter using books which has been given more ratings
y=filtered_ratings.groupby('Book-Title').count()['Book-Rating']>=50
#i have seprated books which has been given more than 50 ratings 
famous_books=y[y].index
final_ratings=filtered_ratings[filtered_ratings['Book-Title'].isin(famous_books)]

In [30]:
#final_ratings.drop_duplicates()
#we are gonna put pivot table on it which will convert the data into rowsxcolumns
pt=final_ratings.pivot_table(index='Book-Title',columns='User-ID',values='Book-Rating')
pt.fillna(0,inplace=True)


In [31]:
from sklearn.metrics.pairwise import cosine_similarity
#in this case the way cosine similarity helps us is for example some users have given ratings 7, 8 ,9 and some have given 3,4,5 cosine similarity will give it perfact match, why??, because their vectors (from pt) point in the same direction
similarity_score=cosine_similarity(pt)



In [32]:
#we will make a function named recommen , jis ke input mein book ka naam hoga aur wo return mein mujhe 5 recommendatiosn dega

def recommend(book_name):
    index=np.where(pt.index==book_name)[0][0]#locating index of the book
    similar_items=sorted(list(enumerate(similarity_score[index])),key=lambda x:x[1],reverse=True)[1:6]
    #what this statement does is it will find the similairty socre of the book with the rest of it, will give us a sorted list with greater scores at the top, why i used lambda here is cuz it sorted on the based of index, i wanted it to sort on the basis of next thing, which were scores, then i only kept the first six books greatest similairity score
    data=[]
    for i in similar_items:
        item=[]
        print(pt.index[i[0]])
        temp_df=books[books['Book-Title']==pt.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Auhtor'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-URL-M'].values))
        
        
        data.append(item)
    return data    
    # the zero here s for index, it will give index and names of books    
        

In [33]:
import pickle 
pickle.dump(popular_df,open('popular.pkl','wb'))

In [34]:
popular_df.head()


,Book-Title,author,Image-URL-M,num_ratings,avg_ratings
0,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,http://images.amazon.com/images/P/0439136350.0...,428,5.852804
3,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,http://images.amazon.com/images/P/0439139597.0...,387,5.824289
5,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,http://images.amazon.com/images/P/0590353403.0...,278,5.737410
9,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,http://images.amazon.com/images/P/043935806X.0...,347,5.501441
13,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,http://images.amazon.com/images/P/0439064872.0...,556,5.183453


In [35]:
import pickle
pickle.dump(pt, open('pt.pkl', 'wb'))  
pickle.dump(books,open('books.pkl','wb'))
pickle.dump(similarity_score,open('similarity_scores.pkl','wb'))